### Generating text with a pretrained llm

In [1]:
import sys
sys.path.append('..')
from downloading_the_base_model.download_model import downlaod_model
downlaod_model(
    repo_id="Qwen/Qwen3-0.6B-Base",
    local_dir="qwen",
)

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 394.86it/s]


'C:\\Users\\Admin\\Documents\\open-posttraining-system\\generating_text_with_pre-trained_llm\\qwen'

In [2]:
from pathlib import Path
from base_model.qwen import Qwen3Tokenizer

tokenizer_path = Path("qwen") / "tokenizer.json"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [3]:
tokenizer

In [4]:
prompt = "Explain neural networks in simple terms."
token_list = tokenizer.encode(prompt=prompt)
token_list

[840, 20772, 29728, 14155, 304, 4285, 3793, 13]

In [5]:
for i in token_list:
    print(f"{i} ---> {tokenizer.decode([i])}")

840 ---> Ex
20772 ---> plain
29728 --->  neural
14155 --->  networks
304 --->  in
4285 --->  simple
3793 --->  terms
13 ---> .


In [6]:
text = tokenizer.decode(token_list)
print(text)

Explain neural networks in simple terms.


### loading the pretraind model

In [7]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU")

GPU available: NVIDIA GeForce RTX 3050 Laptop GPU


In [8]:
# Diagnostic info
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("To use GPU, you need to:")
    print("1. Have an NVIDIA GPU installed")
    print("2. Install NVIDIA CUDA drivers from https://developer.nvidia.com/cuda-downloads")
    print("3. Reinstall PyTorch with CUDA support: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU


In [11]:
from pathlib import Path
from safetensors.torch import load_file
from base_model.qwen import Qwen3Model, load_hf_weights_into_qwen, QWEN_CONFIG_06_B

weights_path = Path("qwen") / "model.safetensors"

# Load ONCE
weights = load_file(weights_path)

model = Qwen3Model(QWEN_CONFIG_06_B)

load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN_CONFIG_06_B["n_layers"],
        "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"]
    },
    params=weights
)

model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()



Qwen3Model(
  (emb): Embedding(151936, 1024)
  (t_block): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=True)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=True)
        (w_values): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=True)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

### ok, now understanding the sequential llm text generation process.